In [1]:
from astropy.time import Time
import astropy.units as u
import json
import pandas as pd

In [2]:
# Function to read the JSON settings file
def get_config():
  
  # Get settings' directory
  settings_dir = r"../config/settings.json"

  # Open JSON file
  with open(settings_dir, "r") as file:
      settings = json.load(file)
  return settings

# Function to import local classification database
def get_localdatabase():
  
  # Data directory
  data_dir = r"../data/body_classification.csv"
  
  # Read CSV file
  return pd.read_csv(data_dir, delimiter=',')

In [17]:
from astroquery.jplhorizons import Horizons
from astroquery.mpc import MPC
import re

def jpl_eph(body, epoch):
  
  # Get ephemerides
  body = Horizons(id=body, epochs=epoch)
  eph = body.ephemerides()
  
  # Extract important fields
  data = eph['datetime_jd', 'RA', 'DEC', 'delta']
  error = eph['RA_3sigma', 'DEC_3sigma', 'SMAA_3sigma', 'SMIA_3sigma', 'Theta_3sigma']
  
  if all(error['RA_3sigma'].mask):
    print('Uncertainty is masked, orbit solution might only be nominal and not have an uncertainty available.')
  
  # Convert to dataframe
  df = data.to_pandas()
  
  # Return data and error
  return df, error

# Function to get ephemerides from MPC
def mpc_eph(body, epoch):
  
  # Handle minutes
  if re.search(r'm$', epoch['step']):
    epoch['step'] = re.sub(r'm$', 'min', epoch['step'])
  
  # Get ephemerides
  eph = MPC.get_ephemeris(
    body, 
    start=epoch['start'], 
    step=epoch['step'], 
    number=epoch['number']
  )
  eph['Date_jd'] = Time(eph['Date']).jd1 # date must be in JD
  
  # Extract important fields
  data = eph['Date_jd', 'RA', 'Dec', 'Delta']
  error = None
  
  # Check for uncertainty fields
  unc_cols = ['Uncertainty 3sig', 'Unc. P.A.']
  if all(col in eph.colnames for col in unc_cols):
    error = eph['Uncertainty 3sig', 'Unc. P.A.']
  else:
    print('Ephemeris uncertainty is not available.')
  
  # Convert to dataframe
  df = data.to_pandas()
  
  # Return data and error
  return df, error

In [19]:
settings = get_config()

rock = settings['body']['id']
epoch = {
      "start": str(Time.now()),
      "stop": str(Time.now() + 1 * u.day),
      "step": "1m",
      "number": 720
    }

eph, err = mpc_eph(rock, epoch)

In [6]:
asteroids = get_localdatabase()
settings = get_config()

families = settings['body']['fields']['family'] or ['is_neo', 'is_mca', 'is_mba', 'is_tjn', 'is_cen', 'is_tno', 'is_paa', 'is_hya', 'is_ast']
states = settings['body']['fields']['state'] or ['is_provisional', 'is_numbered']
obj_unc = settings['body']['fields']['orbit_uncertainty'] or asteroids['orbit_uncertainty'].unique()
limit = settings['body']['fields']['limit'] or 300

filtered_asteroids = asteroids[
  asteroids[families].any(axis=1) &
  asteroids[states].any(axis=1) &
  asteroids['orbit_uncertainty'].isin(obj_unc)
]['designation']

if not filtered_asteroids.empty:
  filtered_asteroids = filtered_asteroids.sample(n=limit).to_list()

filtered_asteroids

['(511743) 2015 DE119',
 '(635186) 2013 BE1',
 '(350164) 2011 TB6',
 '(199763) Davidgregory',
 '(216020) 2005 UD510',
 '(257118) 2008 GS81',
 '(74019) 1998 GY',
 '(362859) 2012 BS54',
 '(426878) 2013 WR37',
 '(692564) 2014 YZ77',
 '(563691) 2016 EO10',
 '(732769) 2014 KO77',
 '(117458) 2005 AN71',
 '(70457) 1999 TU23',
 '(124086) 2001 HZ11',
 '(109511) 2001 QY235',
 '(271892) 2004 VU28',
 '(428992) 2009 AR30',
 '(628512) 2015 RL51',
 '(545177) 2011 BS65',
 '(226472) 2003 SD174',
 '(21535) 1998 OX13',
 '(107865) 2001 FO82',
 '(810623) 2021 FU5',
 '(5080) Oja',
 '(503509) 2016 FQ3',
 '(78412) 2002 QF29',
 '(283796) 2003 SD37',
 '(868554) 2016 EV157',
 '(835680) 2011 SF330',
 '(470985) 2009 SL59',
 '(440528) 2005 UN99',
 '(704424) 2008 GW176',
 '(371257) 2006 BP224',
 '(282370) 2003 PR9',
 '(545366) 2011 GM75',
 '(185335) 2006 VO35',
 '(226010) 2002 EF50',
 '(177675) 2005 ED98',
 '(7857) Lagerros',
 '(250871) 2005 UA431',
 '(89174) 2001 UO55',
 '(811227) 2022 GY10',
 '(857886) 2012 TP230'

In [8]:
# Functions to standardize input data

# Helper function to search body names
def _search_bodies(fields):
  
  # Default direct options
  obj_type = fields['object_type'] or 'asteroid'
  
  if obj_type == 'asteroid':
    asteroids = get_localdatabase()
    
    families = fields['family'] or ['is_neo', 'is_mca', 'is_mba', 'is_tjn', 'is_cen', 'is_tno', 'is_paa', 'is_hya', 'is_ast']
    states = fields['state'] or ['is_provisional', 'is_numbered']
    obj_unc = fields['orbit_uncertainty'] or asteroids['orbit_uncertainty'].unique()
    limit = fields['limit'] or 300
    
    asteroids = asteroids[
      asteroids[families].any(axis=1) &
      asteroids[states].any(axis=1) &
      asteroids['orbit_uncertainty'].isin(obj_unc)
    ]['designation']
      
    if not asteroids.empty:
      asteroids = asteroids.sample(n=limit).to_list()
      
    return asteroids

# Function to standardize input data
def default(settings):
  
  # Default values
  if not settings['limit_magnitude']:
    settings['limit_magnitude'] = 16
  if not settings['exposition_time']:
    settings['exposition_time'] = 5
  if not settings['database']:
    settings['database'] = ["JPL", "MPC"]
  if not settings['observer']['code'] and not settings['observer']['coord']:
    settings['observer']['code'] = "geo"
  if not settings['epoch']:
    settings['epoch'] = {
      "range": {
        "start": str(Time.now()),
        "stop": str(Time.now() + 1 * u.day),
        "step": "1m",
        "number": 720
      }
    }
  if not settings['body']['id']:
    settings['body']['id'] = _search_bodies(settings['body']['fields'])
    
  return settings

In [9]:
settings = get_config()
print(settings)

{'body': {'id': None, 'fields': {'object_type': 'asteroid', 'critical_list_numbered_object': None, 'limit': None, 'family': None, 'orbit_uncertainty': None, 'state': ['is_numbered']}}, 'database': ['MPC'], 'epoch': None, 'observer': {'code': 'G37', 'coord': None}, 'limit_magnitude': None, 'exposition_time': None, 'ADS_key': None}


In [10]:
settings = default(settings)
print(settings)

{'body': {'id': ['(136782) 1996 VJ26', '(61425) 2000 QA16', '(849461) 2005 XM135', '(557634) 2014 WQ83', '(528347) 2008 SV221', '(276751) 2004 FG90', '(761576) 2010 CG26', '(409502) 2005 SZ234', '(249094) 2007 VX161', '(43244) 2000 AR253', '(296322) 2009 ER14', '(194440) 2001 VU107', '(26868) 1993 RS3', '(302468) 2002 EL145', '(201952) 2004 JS43', '(312119) 2007 TY202', '(267717) 2003 BZ47', '(167958) 2005 EX248', '(128130) 2003 QU44', '(402120) 2004 CJ55', '(844518) 2017 BN40', '(493416) 2014 WD218', '(892629) 2015 RX393', '(547632) 2010 TS209', '(396696) 2002 TZ220', '(867002) 2015 TZ474', '(97737) 2000 HX27', '(451199) 2009 UC8', '(750791) 2014 WS574', '(434168) 2002 TM80', '(435728) Yunlin', '(234571) 2001 XO107', '(323462) 2004 JM28', '(857251) 2012 FD96', '(889273) 2011 UV65', '(173533) 2000 WX42', '(799520) 2013 PK115', '(653992) 2014 WH205', '(813845) 2007 TA471', '(635835) 2014 EP143', '(892802) 2015 XX5', '(457029) 2008 CD159', '(501429) 2013 YH149', '(600371) 2011 UN360', '(